# Warm-season UTCI trends in Bangladesh (2001–2020)

The input contains annual April–June UTCI values for each grid.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pymannkendall openpyxl

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymannkendall as mk

input_file = "/content/drive/MyDrive/Bangladesh/Bangladesh_UTCI_FINAL.csv"

output_dir = "/content/drive/MyDrive/Bangladesh/Bangladesh_UTCI_Trend_2001_2020"

os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Read the grid data and check the annual observations
df = pd.read_csv(input_file)

years = list(range(2001, 2021))
utci_cols = [f"UTCI_{y}" for y in years]

required_cols = [
    "Common_ID",
    "Longitude",
    "Latitude",
    "District",
    "Division"
] + utci_cols

missing_cols = [
    c for c in required_cols if c not in df.columns
]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

if df["Common_ID"].isna().any():
    raise ValueError("Missing Common_ID values.")

if df[["District", "Division"]].isna().any().any():
    raise ValueError("Missing district or division names.")

if df["Common_ID"].duplicated().any():
    raise ValueError("Duplicate Common_ID detected.")

df[utci_cols] = df[utci_cols].apply(
    pd.to_numeric, errors="coerce"
)

data = df[utci_cols].to_numpy(dtype=float)

if not np.isfinite(data).all():
    raise ValueError(
        "Missing or infinite UTCI observations detected. "
        "Check the input dataset."
    )

print(f"{len(df):,} grids with {len(years)} annual observations")

In [ ]:
# Calculate Mann–Kendall trends and Sen’s slopes for each grid
sen_slope = np.empty(len(df))
z_values = np.empty(len(df))
p_values = np.empty(len(df))
trends = []

for i in range(len(df)):

    if i % 10000 == 0:
        print(f"Processing {i}/{len(df)}")

    ts = data[i, :]

    result = mk.original_test(ts, alpha=0.05)

    sen_slope[i] = result.slope
    z_values[i] = result.z
    p_values[i] = result.p
    trends.append(result.trend)

df["Sen_slope"] = sen_slope
df["MK_Z"] = z_values
df["MK_p"] = p_values
df["MK_trend"] = trends

# Classify significant trends

df["Increasing_sig"] = (
    (df["MK_trend"] == "increasing") &
    (df["MK_p"] < 0.05)
)

df["Decreasing_sig"] = (
    (df["MK_trend"] == "decreasing") &
    (df["MK_p"] < 0.05)
)

df["Trend_Class"] = "Not significant"

df.loc[
    df["Increasing_sig"], "Trend_Class"
] = "Increasing"

df.loc[
    df["Decreasing_sig"], "Trend_Class"
] = "Decreasing"

In [ ]:
# Summarize trends across the analyzed domain
national_summary = pd.DataFrame({
    "Trend": [
        "Increasing",
        "Decreasing",
        "Not significant"
    ],
    "Number_of_Grids": [
        df["Increasing_sig"].sum(),
        df["Decreasing_sig"].sum(),
        (df["MK_p"] >= 0.05).sum()
    ]
})

national_summary["Percentage"] = (
    national_summary["Number_of_Grids"] /
    len(df) * 100
)

display(national_summary.round(3))

In [ ]:
# Summarize significant trends by district and division
# Percentages use all valid grids; slope statistics use significant grids only.
def summarize_trends(data, group_column):

    results = []

    for name, group in data.groupby(group_column):

        total = len(group)

        inc = group.loc[
            group["Increasing_sig"], "Sen_slope"
        ].dropna()

        dec = group.loc[
            group["Decreasing_sig"], "Sen_slope"
        ].dropna()

        results.append({

            group_column: name,

            "Total_Grids": total,

            "Increasing_Grids_Pct":
                len(inc) / total * 100,

            "Increasing_Min_Slope":
                inc.min() if len(inc) else np.nan,

            "Increasing_Mean_Slope":
                inc.mean() if len(inc) else np.nan,

            "Increasing_Max_Slope":
                inc.max() if len(inc) else np.nan,

            "Decreasing_Grids_Pct":
                len(dec) / total * 100,

            "Decreasing_Min_Slope":
                dec.min() if len(dec) else np.nan,

            "Decreasing_Mean_Slope":
                dec.mean() if len(dec) else np.nan,

            "Decreasing_Max_Slope":
                dec.max() if len(dec) else np.nan

        })

    return pd.DataFrame(results)

district_summary = summarize_trends(
    df, "District"
).sort_values("District")

division_summary = summarize_trends(
    df, "Division"
).sort_values("Division")

display(division_summary.round(3))
display(district_summary.round(3))

In [ ]:
# Check the summaries and inspect the largest slopes
assert national_summary["Number_of_Grids"].sum() == len(df)
for summary in (district_summary, division_summary):
    assert summary["Total_Grids"].sum() == len(df)
    for direction in ("Increasing", "Decreasing"):
        count = (summary[f"{direction}_Grids_Pct"] * summary["Total_Grids"] / 100).sum()
        assert np.isclose(count, df[f"{direction}_sig"].sum())

print("Districts:", len(district_summary))
print("Divisions:", len(division_summary))
print("Districts with significant increases:", (district_summary["Increasing_Grids_Pct"] > 0).sum())
print("Districts with significant decreases:", (district_summary["Decreasing_Grids_Pct"] > 0).sum())
print("Maximum Sen's slope:", df["Sen_slope"].max())
print("Minimum Sen's slope:", df["Sen_slope"].min())

trend_fields = ["Common_ID", "District", "Division", "Sen_slope", "MK_Z", "MK_p", "MK_trend"]
display(df.nlargest(10, "Sen_slope")[trend_fields])

# Inspect the six grids examined during the initial analysis.
target_ids = [83693, 1604, 18766, 78975, 108465, 61256]
display(df.loc[df["Common_ID"].isin(target_ids), trend_fields])

In [ ]:
# Save the grid results for joining to the original geometry in ArcGIS Pro
grid_trend = df[
    [
        "Common_ID",
        "Longitude",
        "Latitude",
        "District",
        "Division",
        "Sen_slope",
        "MK_Z",
        "MK_p",
        "MK_trend",
        "Trend_Class"
    ]
].copy()

grid_trend.to_csv(
    f"{output_dir}/Grid_Level_UTCI_Trends_2001_2020.csv",
    index=False
)

# Keep the full input attributes alongside the recalculated trends.
df.to_csv(os.path.join(output_dir, "Bangladesh_UTCI_trend_2001_2020.csv"), index=False)

# Save the national and administrative tables
national_summary.to_csv(f"{output_dir}/National_Trend_Summary.csv", index=False)
district_summary.to_csv(f"{output_dir}/Table_S1_District_Trend_2001_2020.csv", index=False)
division_summary.to_csv(f"{output_dir}/Table_1_Division_Trend_2001_2020.csv", index=False)

excel_path = (
    f"{output_dir}/Bangladesh_UTCI_Trend_Tables_2001_2020.xlsx"
)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    national_summary.to_excel(
        writer,
        sheet_name="National_Trends",
        index=False
    )

    division_summary.to_excel(
        writer,
        sheet_name="Table_1_Division",
        index=False
    )

    district_summary.to_excel(
        writer,
        sheet_name="Table_S1_District",
        index=False
    )

print("Tables and grid results saved to:", output_dir)

In [ ]:
# Figure S1: district-level trend distribution
plot_data = district_summary.sort_values(
    "Increasing_Grids_Pct",
    ascending=True
).copy()

increasing = plot_data[
    "Increasing_Grids_Pct"
].values

decreasing = plot_data[
    "Decreasing_Grids_Pct"
].values

nonsignificant = 100 - increasing - decreasing

x = np.arange(len(plot_data))

fig, ax = plt.subplots(figsize=(17, 7))

ax.bar(
    x,
    increasing,
    color="#C94C3C",
    label="Increasing",
    width=0.72
)

ax.bar(
    x,
    nonsignificant,
    bottom=increasing,
    color="#E9A45B",
    label="No statistically significant trend",
    width=0.72
)

ax.bar(
    x,
    decreasing,
    bottom=increasing + nonsignificant,
    color="#477BA7",
    label="Decreasing",
    width=0.72
)

ax.set_xticks(x)

ax.set_xticklabels(
    plot_data["District"],
    rotation=65,
    ha="right",
    fontsize=8
)

ax.set_ylim(0, 105)

ax.set_ylabel(
    "Percentage of analyzed grid cells (%)",
    fontsize=11
)

ax.set_xlabel(
    "District",
    fontsize=11
)

ax.set_title(
    "Mann–Kendall Trend Distribution Across Districts (2001–2020)",
    fontsize=13,
    pad=12
)

ax.set_yticks(np.arange(0, 101, 20))

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

ax.set_axisbelow(True)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.27),
    ncol=3,
    frameon=False,
    fontsize=10
)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()

plt.savefig(
    f"{output_dir}/Figure_S1_District_MK_2001_2020.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{output_dir}/Figure_S1_District_MK_2001_2020.pdf",
    bbox_inches="tight"
)

plt.show()